# Results paragraph: Developmental (age) effects on microstructural measures

Uses assembled **age–quality effects** (no_quality only): ΔR²adj = full GAMM (age, sex, site) minus reduced model excluding age. Data: `data/age_effects/age_effects_all_outputs.rds`. Filter: `qc_metric == "no_quality"`, harmonized, pooled, five metrics (ICVF, MKT, RTOP, FA, MD).

**Fig. 4:** Panel A = age-effect heatmap (ΔR²adj); Panel B = ICVF; Panel C = MKT/RTOP. **Supplementary Fig. S11** = all measures.

## Setup and load age effects

In [4]:
suppressPackageStartupMessages({
  library(dplyr)
  library(fs)
  library(jsonlite)
})

config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json.")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)

age_rds <- fs::path(project_root, "data", "age_effects", "age_effects_all_outputs.rds")
if (!file.exists(age_rds)) stop("Age effects file not found: ", age_rds)

df_all <- readRDS(age_rds)
if (!is.data.frame(df_all)) stop("Age effects file is not a data.frame.")

qc_target <- "no_quality"
source_target <- "harmonized"
output_target <- "pooled"
metrics_keep <- c("DKI_mkt", "NODDI_icvf", "MAPMRI_rtop", "GQI_fa", "GQI_md")
metric_labels <- c(
  DKI_mkt = "MKT",
  NODDI_icvf = "ICVF",
  MAPMRI_rtop = "RTOP",
  GQI_fa = "FA",
  GQI_md = "MD"
)

qc_col <- if ("qc_metric" %in% names(df_all)) "qc_metric" else "qc_covariate"
if (!qc_col %in% names(df_all) || !qc_target %in% df_all[[qc_col]]) stop("No no_quality rows in age effects.")
df <- df_all %>%
  filter(
    .data[[qc_col]] == qc_target,
    .data[["source"]] == source_target,
    .data[["output_type"]] == output_target,
    metric %in% metrics_keep,
    !is.na(.data[["age_effect_size"]])
  )

cat("N rows (no_quality, harmonized, pooled, five metrics):", nrow(df), "\n")

N rows (no_quality, harmonized, pooled, five metrics): 310 


## Age effect (ΔR²adj) summary by metric

Min, max, and mean of age effect size (as % variance explained) across bundles, for paragraph placeholders.

In [5]:
# age_effect_size is ΔR²adj (proportion); convert to % for paragraph
effect_summary <- df %>%
  mutate(effect_pct = .data[["age_effect_size"]] * 100) %>%
  group_by(metric) %>%
  summarise(
    min_pct = min(effect_pct, na.rm = TRUE),
    max_pct = max(effect_pct, na.rm = TRUE),
    mean_pct = mean(effect_pct, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  mutate(metric_label = unname(metric_labels[metric])) %>%
  arrange(desc(mean_pct))

cat("Age effect (ΔR²adj as % variance) across bundles by metric:\n")
print(effect_summary)

# One decimal for range/mean in text; format range as X.X–X.X or X–X
fmt_range <- function(mn, mx) {
  if (length(mn) < 1L || length(mx) < 1L || is.na(mn[1]) || is.na(mx[1])) return("—")
  mn <- mn[1]; mx <- mx[1]
  if (mn >= 10 || mx >= 10) sprintf("%.0f–%.0f", mn, mx) else sprintf("%.1f–%.1f", mn, mx)
}
fmt_mean <- function(x) if (length(x) < 1L || is.na(x[1])) "—" else sprintf("%.1f", x[1])

get_row <- function(met) {
  effect_summary %>% filter(metric == met) %>% slice(1)
}

icvf  <- get_row("NODDI_icvf")
mkt   <- get_row("DKI_mkt")
rtop  <- get_row("MAPMRI_rtop")
md    <- get_row("GQI_md")
fa    <- get_row("GQI_fa")

icvf_range  <- fmt_range(icvf$min_pct,  icvf$max_pct)
icvf_mean   <- fmt_mean(icvf$mean_pct)
mkt_range   <- fmt_range(mkt$min_pct,   mkt$max_pct)
mkt_mean    <- fmt_mean(mkt$mean_pct)
rtop_range  <- fmt_range(rtop$min_pct,  rtop$max_pct)
rtop_mean   <- fmt_mean(rtop$mean_pct)
md_range    <- fmt_range(md$min_pct,   md$max_pct)
md_mean     <- fmt_mean(md$mean_pct)
fa_range    <- fmt_range(fa$min_pct,   fa$max_pct)
fa_mean     <- fmt_mean(fa$mean_pct)

# Bundle-category sensitivity sentence for paragraph
category_clause <- "association bundles exhibiting the strongest age effects, and cerebellar bundles the weakest"
if ("bundle_category" %in% names(df)) {
  cat_map <- c(
    "Association" = "association",
    "ProjectionBasalGanglia" = "subcortical",
    "ProjectionBrainstem" = "brainstem",
    "Cerebellum" = "cerebellar",
    "Commissure" = "commissural"
  )
  cat_rank <- df %>%
    mutate(cat_pretty = dplyr::coalesce(unname(cat_map[as.character(bundle_category)]), as.character(bundle_category))) %>%
    group_by(cat_pretty) %>%
    summarise(mean_effect_pct = mean(age_effect_size, na.rm = TRUE) * 100, .groups = "drop") %>%
    arrange(desc(mean_effect_pct))
  if (nrow(cat_rank) >= 2) {
    strongest <- as.character(cat_rank$cat_pretty[1])
    weakest <- as.character(cat_rank$cat_pretty[nrow(cat_rank)])
    if (!(strongest == "association" && weakest == "cerebellar")) {
      category_clause <- paste0("association bundles exhibiting the strongest age effects, and ", weakest, " bundles the weakest")
    }
  }
}

cat("\nParagraph placeholders:\n")
cat("  ICVF:  range", icvf_range, "%, mean", icvf_mean, "%\n")
cat("  MKT:   range", mkt_range, "%, mean", mkt_mean, "%\n")
cat("  RTOP:  range", rtop_range, "%, mean", rtop_mean, "%\n")
cat("  MD:    range", md_range, "%, mean", md_mean, "%\n")
cat("  FA:    range", fa_range, "%, mean", fa_mean, "%\n")

Age effect (ΔR²adj as % variance) across bundles by metric:
# A tibble: 5 × 5
  metric      min_pct max_pct mean_pct metric_label
  <chr>         <dbl>   <dbl>    <dbl> <chr>       
1 NODDI_icvf   5.82      52.1    34.5  ICVF        
2 DKI_mkt      4.62      48.5    32.6  MKT         
3 MAPMRI_rtop  0.561     49.1    28.7  RTOP        
4 GQI_md       0.147     27.3    12.5  MD          
5 GQI_fa       0.0201    17.4     7.43 FA          

Paragraph placeholders:
  ICVF:  range 6–52 %, mean 34.5 %
  MKT:   range 5–49 %, mean 32.6 %
  RTOP:  range 1–49 %, mean 28.7 %
  MD:    range 0–27 %, mean 12.5 %
  FA:    range 0–17 %, mean 7.4 %


## Filled paragraph

In [6]:
txt <- paste(
  "Across microstructural metrics, the strongest developmental effects were observed for ICVF (Fig. 4b), followed by MKT and RTOP (Fig. 4c).",
  "ICVF explained", icvf_range, "% of variance across bundles (mean ΔR²adj across bundles =", icvf_mean, "%),",
  "MKT explained", mkt_range, "% (mean =", mkt_mean, "%), and RTOP explained", rtop_range, "% (mean =", rtop_mean, "%).",
  "In contrast, tensor-derived measures showed substantially weaker developmental sensitivity: mean diffusivity (MD) explained", md_range, "% of variance (mean =", md_mean, "%), and fractional anisotropy (FA) explained", fa_range, "% (mean =", fa_mean, "%).",
  "Developmental sensitivity also varied across bundle categories, with", category_clause, ".",
  "Developmental effects for all microstructural metrics are provided in Supplementary Fig. S11.",
  "Together, these results emphasize that advanced multi-compartment and non-Gaussian metrics are more sensitive to developmental effects than conventional tensor-derived measures.",
  sep = " "
)
cat(txt, "\n")

Across microstructural metrics, the strongest developmental effects were observed for ICVF (Fig. 4b), followed by MKT and RTOP (Fig. 4c). ICVF explained 6–52 % of variance across bundles (mean ΔR²adj across bundles = 34.5 %), MKT explained 5–49 % (mean = 32.6 %), and RTOP explained 1–49 % (mean = 28.7 %). In contrast, tensor-derived measures showed substantially weaker developmental sensitivity: mean diffusivity (MD) explained 0–27 % of variance (mean = 12.5 %), and fractional anisotropy (FA) explained 0–17 % (mean = 7.4 %). Developmental sensitivity also varied across bundle categories, with association bundles exhibiting the strongest age effects, and cerebellar bundles the weakest . Developmental effects for all microstructural metrics are provided in Supplementary Fig. S11. Together, these results emphasize that advanced multi-compartment and non-Gaussian metrics are more sensitive to developmental effects than conventional tensor-derived measures. 
